# 2 — Acesso e Inclusão

Objetivo deste noteboo é realizar a ingestão, o recorte territorial para o município de Porto Alegre (código IBGE 4314902 / 431490) e o diagnóstico estrutural das bases tabulares e geoespaciais do projeto.

Fontes analisadas:
• Demografia e Renda (Censo 2022 - IBGE): Tabelas agregadas por setor censitário (básico, demografia e renda do responsável).
• Saúde (CNES): Cadastro Nacional de Estabelecimentos de Saúde.
• Educação (INEP/Secretaria): Lista detalhada das escolas de Porto Alegre.
• Assistência Social: Camada oficial de equipamentos socioassistenciais do município.
• Território e Vulnerabilidade (IBGE): Malha de setores censitários do RS e polígonos de Favelas e Comunidades Urbanas (FCUs).

In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import pyogrio
from charset_normalizer import from_path
from docx import Document

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "1_raw"

print("Repositório:", ROOT)
print("Dados brutos:", RAW)

Repositório: /Users/aluno-81/Documents/projects/eld_dataset
Dados brutos: /Users/aluno-81/Documents/projects/eld_dataset/data/1_raw


In [2]:
def detectar_encoding(caminho):
    melhor = from_path(caminho).best()
    return melhor.encoding if melhor else "desconhecido"


def carregar_csv(nome, separador=";"):
    caminho = RAW / nome
    encoding = detectar_encoding(caminho)
    df = pd.read_csv(
        caminho,
        sep=separador,
        encoding=encoding,
        dtype=str,
    )
    return df, encoding, separador


In [3]:
basico, enc_basico, sep_basico = carregar_csv("Agregados_por_setores_basico_BR.csv")
demografia, enc_demografia, sep_demografia = carregar_csv("Agregados_por_setores_demografia_BR.csv")
renda, enc_renda, sep_renda = carregar_csv("Agregados_por_setores_renda_responsavel_BR.csv")
cnes, enc_cnes, sep_cnes = carregar_csv("cnes_estabelecimentos.csv")

escolas, enc_escolas, sep_escolas = carregar_csv(
    "Análise - Tabela da lista das escolas - Detalhado.csv",
    separador=",",
)
escolas.columns = [col.lstrip("\ufeff") for col in escolas.columns]

print("Bases tabulares carregadas:")
print("- Básico IBGE:", basico.shape, enc_basico, sep_basico)
print("- Demografia IBGE:", demografia.shape, enc_demografia, sep_demografia)
print("- Renda IBGE:", renda.shape, enc_renda, sep_renda)
print("- CNES:", cnes.shape, enc_cnes, sep_cnes)
print("- Escolas:", escolas.shape, enc_escolas, sep_escolas)


Bases tabulares carregadas:
- Básico IBGE: (468099, 38) cp1250 ;
- Demografia IBGE: (458772, 37) ascii ;
- Renda IBGE: (458772, 7) ascii ;
- CNES: (643594, 36) cp1250 ;
- Escolas: (988, 19) utf_8 ,


In [4]:
setores = gpd.read_file(
    RAW / "RS_setores_CD2022.gpkg",
    layer="RS_setores_CD2022",
)

assistencia_camada = pyogrio.list_layers(RAW / "assistencia_social_poa.gpkg")[0][0]
assistencia = gpd.read_file(
    RAW / "assistencia_social_poa.gpkg",
    layer=assistencia_camada,
)

fcu = gpd.read_file(
    RAW / "poligonos_FCUs_shp" / "qg_2022_670_fcu_agreg.shp"
)

print("Bases espaciais carregadas:")
print("- Setores IBGE:", setores.shape, setores.crs)
print("- Assistência social:", assistencia.shape, assistencia.crs, "| camada:", assistencia_camada)
print("- FCUs:", fcu.shape, fcu.crs)

cpg = RAW / "poligonos_FCUs_shp" / "qg_2022_670_fcu_agreg.cpg"
print("- CPG do shapefile:", cpg.read_text(encoding="ascii").strip())


Bases espaciais carregadas:
- Setores IBGE: (25575, 37) EPSG:4674
- Assistência social: (35, 6) EPSG:4674 | camada: assistencia_social_poa_100_oficial
- FCUs: (12348, 8) EPSG:4674
- CPG do shapefile: UTF-8


In [5]:
# Recortes em memória. Nenhuma linha é removida da fonte carregada.
basico_poa = basico.loc[basico["CD_MUN"].astype("string") == "4314902"]
demografia_poa = demografia.loc[
    demografia["CD_setor"].astype("string").str.startswith("4314902", na=False)
]
renda_poa = renda.loc[
renda["CD_SETOR"].astype("string").str.startswith("4314902", na=False)
]
cnes_poa = cnes.loc[cnes["CO_IBGE"].astype("string") == "431490"]

setores_poa = setores.loc[
    setores["CD_MUN"].astype("string") == "4314902"
    if "CD_MUN" in setores.columns
    else setores["CD_SETOR"].astype("string").str.startswith("4314902", na=False)
]
fcu_poa = fcu.loc[fcu["cd_mun"].astype("string") == "4314902"]

# A base de escolas já contém apenas Porto Alegre.
escolas_poa = escolas.loc[escolas["Município"].astype("string") == "Porto Alegre"]

# O GPKG de assistência já representa Porto Alegre.
assistencia_poa = assistencia.copy()

summary_dimensoes = pd.DataFrame([
    {"Base": "IBGE básico CSV", "Antes": basico.shape, "Depois": basico_poa.shape, "Filtro": "CD_MUN = 4314902"},
    {"Base": "IBGE demografia CSV", "Antes": demografia.shape, "Depois": demografia_poa.shape, "Filtro": "CD_setor inicia 4314902"},
    {"Base": "IBGE renda CSV", "Antes": renda.shape, "Depois": renda_poa.shape, "Filtro": "CD_SETOR inicia 4314902"},
    {"Base": "CNES CSV", "Antes": cnes.shape, "Depois": cnes_poa.shape, "Filtro": "CO_IBGE = 431490"},
    {"Base": "Escolas CSV", "Antes": escolas.shape, "Depois": escolas_poa.shape, "Filtro": "Município = Porto Alegre"},
    {"Base": "Setores GPKG", "Antes": setores.shape, "Depois": setores_poa.shape, "Filtro": "município/setor de POA"},
    {"Base": "Assistência GPKG", "Antes": assistencia.shape, "Depois": assistencia_poa.shape, "Filtro": "base já limitada a POA"},
    {"Base": "FCUs SHP", "Antes": fcu.shape, "Depois": fcu_poa.shape, "Filtro": "cd_mun = 4314902"},
])

display(summary_dimensoes)


,Base,Antes,Depois,Filtro
0,IBGE básico CSV,"(468099, 38)","(2744, 38)",CD_MUN = 4314902
1,IBGE demografia CSV,"(458772, 37)","(2713, 37)",CD_setor inicia 4314902
2,IBGE renda CSV,"(458772, 7)","(2713, 7)",CD_SETOR inicia 4314902
3,CNES CSV,"(643594, 36)","(10294, 36)",CO_IBGE = 431490
4,Escolas CSV,"(988, 19)","(988, 19)",Município = Porto Alegre
5,Setores GPKG,"(25575, 37)","(2748, 37)",município/setor de POA
6,Assistência GPKG,"(35, 6)","(35, 6)",base já limitada a POA
7,FCUs SHP,"(12348, 8)","(125, 8)",cd_mun = 4314902


## Formato e organização das bases

In [6]:
formatos = pd.DataFrame([
    {"Arquivo": "Agregados_por_setores_basico_BR.csv", "Tipo": "CSV", "Encoding": enc_basico, "Separador": repr(sep_basico), "CRS": "N/A"},
    {"Arquivo": "Agregados_por_setores_demografia_BR.csv", "Tipo": "CSV", "Encoding": enc_demografia, "Separador": repr(sep_demografia), "CRS": "N/A"},
    {"Arquivo": "Agregados_por_setores_renda_responsavel_BR.csv", "Tipo": "CSV", "Encoding": enc_renda, "Separador": repr(sep_renda), "CRS": "N/A"},
    {"Arquivo": "cnes_estabelecimentos.csv", "Tipo": "CSV", "Encoding": enc_cnes, "Separador": repr(sep_cnes), "CRS": "N/A"},
    {"Arquivo": "Análise - Tabela da lista das escolas - Detalhado.csv", "Tipo": "CSV", "Encoding": enc_escolas, "Separador": repr(sep_escolas), "CRS": "N/A"},
    {"Arquivo": "RS_setores_CD2022.gpkg", "Tipo": "Geoespacial", "Encoding": "N/A", "Separador": "N/A", "CRS": str(setores.crs)},
    {"Arquivo": "assistencia_social_poa.gpkg", "Tipo": "Geoespacial", "Encoding": "N/A", "Separador": "N/A", "CRS": str(assistencia.crs)},
    {"Arquivo": "qg_2022_670_fcu_agreg.shp", "Tipo": "Shapefile", "Encoding": cpg.read_text(encoding="ascii").strip(), "Separador": "N/A", "CRS": str(fcu.crs)},
])

display(formatos)


,Arquivo,Tipo,Encoding,Separador,CRS
0,Agregados_por_setores_basico_BR.csv,CSV,cp1250,';',N/A
1,Agregados_por_setores_demografia_BR.csv,CSV,ascii,';',N/A
2,Agregados_por_setores_renda_responsavel_BR.csv,CSV,ascii,';',N/A
3,cnes_estabelecimentos.csv,CSV,cp1250,';',N/A
4,Análise - Tabela da lista das escolas - Detal...,CSV,utf_8,"','",N/A
5,RS_setores_CD2022.gpkg,Geoespacial,N/A,N/A,EPSG:4674
6,assistencia_social_poa.gpkg,Geoespacial,N/A,N/A,EPSG:4674
7,qg_2022_670_fcu_agreg.shp,Shapefile,UTF-8,N/A,EPSG:4674


In [7]:
colunas_selecionadas = {
    "IBGE básico": (
        basico_poa,
        ["CD_SETOR", "CD_MUN", "v0001", "AREA_KM2", "SITUACAO", "CD_SIT", "CD_TIPO", "CD_FCU", "NM_FCU"],
    ),
    "IBGE demografia": (
        demografia_poa,
        ["CD_setor", "V01006", "V01040", "V01041"],
    ),
    "IBGE renda": (
        renda_poa,
        ["CD_SETOR", "V06001", "V06004", "V06005", "V06006"],
    ),
    "Setores GPKG": (
        setores_poa,
        ["CD_SETOR", "CD_MUN", "v0001", "AREA_KM2", "SITUACAO", "CD_SIT", "CD_TIPO", "CD_FCU", "NM_FCU", "geometry"],
    ),
    "CNES": (
        cnes_poa,
        ["CO_CNES", "CO_UNIDADE", "NO_FANTASIA", "NO_RAZAO_SOCIAL", "CO_IBGE", "TP_UNIDADE", "CO_NATUREZA_JUR", "DS_ESFERA_ADMINISTRATIVA", "CO_AMBULATORIAL_SUS", "CO_MOTIVO_DESAB", "NU_LATITUDE", "NU_LONGITUDE", "ST_ATEND_HOSPITALAR"],
    ),
    "Escolas": (
        escolas_poa,
        ["Município", "Restrição de Atendimento", "Categoria Administrativa", "Dependência Administrativa", "Latitude", "Longitude"],
    ),
    "Assistência GPKG": (
        assistencia_poa,
        ["categoria", "nome", "endereco", "latitude", "longitude", "geometry"],
    ),
    "FCUs SHP": (
        fcu_poa,
        ["cd_fcu", "nm_fcu", "cd_mun", "nm_mun", "geometry"],
    ),
}

for nome, (df, colunas) in colunas_selecionadas.items():
    presentes = [c for c in colunas if c in df.columns]
    ausentes = [c for c in colunas if c not in df.columns]
    print(f"\n== {nome} ==")
    print(df[presentes].dtypes.to_string())
    if ausentes:
        print("Colunas não encontradas:", ausentes)



== IBGE básico ==
CD_SETOR    str
CD_MUN      str
v0001       str
AREA_KM2    str
SITUACAO    str
CD_SIT      str
CD_TIPO     str
CD_FCU      str
NM_FCU      str

== IBGE demografia ==
CD_setor    str
V01006      str
V01040      str
V01041      str

== IBGE renda ==
CD_SETOR    str
V06001      str
V06004      str
V06005      str
V06006      str

== Setores GPKG ==
CD_SETOR         str
CD_MUN           str
v0001          int64
AREA_KM2     float64
SITUACAO         str
CD_SIT           str
CD_TIPO          str
CD_FCU           str
NM_FCU           str
geometry    geometry

== CNES ==
CO_CNES                     str
CO_UNIDADE                  str
NO_FANTASIA                 str
NO_RAZAO_SOCIAL             str
CO_IBGE                     str
TP_UNIDADE                  str
CO_NATUREZA_JUR             str
DS_ESFERA_ADMINISTRATIVA    str
CO_AMBULATORIAL_SUS         str
CO_MOTIVO_DESAB             str
NU_LATITUDE                 str
NU_LONGITUDE                str
ST_ATEND_HOSPITALAR       

In [8]:
arquivos_excel = [
    RAW / "dicionario_de_dados_agregados_por_setores_censitarios_20261113.xlsx",
    RAW / "dicionario_de_dados_renda_responsavel_20260508.xlsx",
    RAW / "SCNES_DOMINIOS.XLS",
]

print("== Dicionários e domínios ==")
for caminho in arquivos_excel:
    try:
        excel = pd.ExcelFile(caminho)
        print(f"\n{caminho.name}")
        for aba in excel.sheet_names:
            df_aba = pd.read_excel(caminho, sheet_name=aba, dtype=str)
            print(f"- {aba}: shape={df_aba.shape}")
    except Exception as exc:
        print(f"- Não abriu: {exc}")

docx_caminho = RAW / "SCNES_DICIONARIO_DE_DADOS" / "DICIONARIO_DE_DADOS.docx"
doc = Document(docx_caminho)
print(f"\n{docx_caminho.name}")
print("- Parágrafos:", len(doc.paragraphs))
print("- Tabelas:", len(doc.tables))
print("- Linhas totais nas tabelas:", sum(len(t.rows) for t in doc.tables))


== Dicionários e domínios ==



dicionario_de_dados_agregados_por_setores_censitarios_20261113.xlsx
- Dicionário Básico: shape=(9, 3)
- Siglas Básico: shape=(7, 2)


- Dicionário não PCT: shape=(1411, 4)
- Dicionário PCT - Indígenas: shape=(1029, 4)


- Dicionário PCT - Quilombolas: shape=(951, 4)
- Dicionário Domicílios Coletivos: shape=(146, 4)

dicionario_de_dados_renda_responsavel_20260508.xlsx
- Dicionário Renda Responsável: shape=(6, 3)



SCNES_DOMINIOS.XLS


- ADESÃO PROG PROJ SAÚDE: shape=(18, 6)


... (28 outputs left out)

- TIPOS DE ESTABELECIMENTO: shape=(40, 2)
- TIPOS DE LEITOS: shape=(7, 2)


- TIPOS DE LOGRADOURO: shape=(269, 2)
- TURNO DE ATENDIMENTO: shape=(7, 2)


- Planilha2: shape=(0, 0)
- Planilha1: shape=(0, 0)



DICIONARIO_DE_DADOS.docx
- Parágrafos: 1368
- Tabelas: 350
- Linhas totais nas tabelas: 2199
